In [1]:
# IMPORTS -------------------------------------------------------------------- #
import os
import sys
from pathlib import Path
import re

import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
import pandas as pd
import seaborn as sns

import utils

workspace,experiment=utils.workspace()
from __future__ import annotations

sys.path = [p for p in sys.path if "positron-python" not in p]

import anndata as ad
import scanpy as sc

In [2]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.set_figure_params(dpi=80, facecolor="white")
sc.logging.print_header()

Package,Version
matplotlib,3.11.1
numpy,2.5.2
pandas,3.0.5
seaborn,0.13.2
anndata,0.13.2
scanpy,1.12.3
Component,Info
Python,"3.14.6 | packaged by conda-forge | (main, Aug 11 2026, 10:26:15) [GCC 14.4.0]"
OS,Linux-6.1.0-53-amd64-x86_64-with-glibc2.36
CPU,256/256 logical CPU cores


In [3]:
# PATHS ---------------------------------------------------------------------- #
results_file = workspace/'data/pbmc3k.h5ad'
read_file=workspace/'data/PBMC3k/filtered_gene_bc_matrices/hg19'
result_dir=experiment/'results'
assert read_file.exists()

In [4]:
# READ 10X-GENOMICS-FORMATTED MTX DIRECTORY ---------------------------------- #
# path: Path to directory for .mtx and .tsv files
# var_names: the variables index (options from genes.tsv file: 'gene_symbols', 'gene_ids')
# make_unique: Whether to make the variables index unique by appending ‘-1’, ‘-2’ etc. or not.
# cache: If False, read from source, if True, read from fast ‘h5ad’ cache.
# ---------------------------------------------------------------------------- #
genes=pd.read_csv(read_file/'genes.tsv',sep='\t',header=None).rename(columns={0:'gene_ids',1:'gene_symbols'})
adata = sc.read_10x_mtx(
    path=read_file,
    var_names='gene_symbols', #  # use gene symbols for the variable names (variables-axis index)
    make_unique=True, # make gene symbols unique
    cache=True, # write a cache file for faster subsequent reading
    )
adata.write(result_dir/'pbmc3k_raw.h5ad')
adata

... reading from cache file cache/omics-odcf-analysis-OE0290_projects-heroes-aya_pools-AG_Thongjuea-Code-10x_multiomics-Selina-project-data-PBMC3k-filtered_gene_bc_matrices-hg19-matrix.h5ad


AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)

In [5]:
adata.var

,gene_ids
MIR1302-10,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092
RP11-34P13.7,ENSG00000238009
RP11-34P13.8,ENSG00000239945
...,...
AC145205.1,ENSG00000215635
BAGE5,ENSG00000268590
CU459201.1,ENSG00000251180
AC002321.2,ENSG00000215616


In [6]:
new_genes = list(set([re.sub(r'-.','',gene) for gene in adata.var[~adata.var.index.isin(genes.gene_symbols)].index]))
len(new_genes)

91

In [7]:
# DUPLICATE GENE SYMBOLS ----------------------------------------------------- #
# during read_10x_mtx, some gene_symbols are already recognized as duplicates
# they receive an appendage: U1 -> [U1, U1-1]
# ---------------------------------------------------------------------------- #

print(adata.var.index.duplicated().sum()) # if 0 no adata.var_names_make_unique() necessary

dupl = [idx.replace('-1','') for idx in adata.var.index if idx.endswith('-1') and idx.replace('-1','') in adata.var.index]
genes_duplicated = genes[genes.gene_symbols.duplicated()].drop_duplicates(subset=['gene_symbols'])
genes_duplicated.shape, len(dupl)

pd_dupl = pd.DataFrame([{'symbol':gene,'genes':genes.query('gene_symbols==@gene').shape[0],'adata':adata.var.reset_index(names='gene_symbols').query('gene_symbols==@gene').shape[0]} for gene in dupl])
pd_dupl

0


,symbol,genes,adata
0,U1,2,1
1,EFNA3,2,1
2,DCAF8,2,1
3,MRPL30,2,1
4,LIMS3,2,1
...,...,...,...
86,ZNF84,2,1
87,ZNF26,2,1
88,KIR3DL2,2,1
89,PNRC2,2,1


# QC metrics

In [8]:
# DEFINE GENES --------------------------------------------------------------- #
# mitochondrial genes
genes_mt = adata.var_names.str.startswith("MT-")
adata.var["mt"] = genes_mt
# ribosomal genes
genes_ribo = adata.var_names.str.startswith(("RPCS","RPL"))
adata.var["ribo"] = genes_ribo
# hemoglobin genes.
genes_hb = adata.var_names.str.contains(r"^HB[ABDEGMQZ]\d*(?!\w)")
adata.var["hb"] = genes_hb
sum(genes_mt),sum(genes_ribo),sum(genes_hb)

(np.int64(13), np.int64(56), np.int64(10))